# SynNat-BERT Atom Attention Weight Visualization

Visualizing the attention mechanism using BertViz (https://github.com/jessevig/bertviz#-paper)

Adapted from ChemBERTa (22_Transfer_Learning_With_ChemBERTa_Transformers.ipynb)

In [1]:
import sys

if not 'bertviz_repo' in sys.path:
  sys.path += ['bertviz_repo']

from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline
from bertviz import head_view

/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
import numpy as np
import torch
from utils_adapt import (
    ChemBERTaDescriptorFusion,
    MoleculeDataset,
    compute_descriptor_matrix,
    GaussianizedPCATransformer,
    prepare_descriptors_with_gaussianized_pca
)
import joblib
import pandas as pd

In [3]:
%%javascript
require.config({
  paths: {
      d3: '//cdnjs.cloudflare.com/ajax/libs/d3/3.4.8/d3.min',
      jquery: '//ajax.googleapis.com/ajax/libs/jquery/2.0.0/jquery.min',
  }
});

<IPython.core.display.Javascript object>

In [4]:
def call_html():
  import IPython
  display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
        <script>
          requirejs.config({
            paths: {
              base: '/static/base',
              "d3": "https://cdnjs.cloudflare.com/ajax/libs/d3/3.5.8/d3.min",
              jquery: '//ajax.googleapis.com/ajax/libs/jquery/2.0.0/jquery.min',
            },
          });
        </script>
        '''))

In [5]:
from bertviz import head_view, model_view
from bertviz.neuron_view import show
from transformers import AutoTokenizer, AutoModel, utils

Load pre-trained / adapted checkpoints to assign attention weights

In [12]:
import json
import os

tokenizer_version = "DeepChem/ChemBERTa-100M-MLM"
model_version = "/code/checkpoints/adapted_SNbert_regre"

config_path = os.path.join(model_version, "model_config.json")
text_encoder_path = os.path.join(model_version, "text_encoder")

with open(config_path, "r", encoding="utf-8") as f:
    model_config = json.load(f)

model_config["pretrained_model_name"] = (
    text_encoder_path if os.path.isdir(text_encoder_path) else tokenizer_version
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2, ensure_ascii=False)

tokenizer = AutoTokenizer.from_pretrained(tokenizer_version)
model = ChemBERTaDescriptorFusion.from_pretrained(model_version)
model.eval()

model.text_encoder.config.output_attentions = True

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading text encoder from /code/checkpoints/adapted_SNbert_regre/text_encoder...


In [13]:
SMILES_a = "C#CC1=CC=CC(N/C2=N/C=N\C3=CC(OCCOC)=C(OCCOC)C=C23)=C1"
desc_df, valid_idx = compute_descriptor_matrix([SMILES_a], verbose=True)
desc_array = desc_df.values.astype(np.float32)

transformer_path = '/code/checkpoints/adapted_SNbert_regre/gaussianized_pca_transformer.pkl'
transformer = joblib.load(transformer_path)
desc_array_scaled = transformer.transform(desc_array)

Computing descriptors: 100%|██████████| 1/1 [00:00<00:00, 938.74it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [14]:
encoded_input = tokenizer([SMILES_a], padding=False, truncation=True)
encoded_input = {k: v for k, v in encoded_input.items()}
attention_mask = encoded_input['attention_mask']

descriptors_tensor = torch.tensor(desc_array_scaled, dtype=torch.float32)
attention_mask = torch.tensor(attention_mask, dtype=torch.float32)

if descriptors_tensor.dim() == 1:
    descriptors_tensor = descriptors_tensor.unsqueeze(0)

with torch.no_grad():
    model_output = model(**encoded_input, desc=descriptors_tensor,
                         output_sequence_output=True, output_attentions=True)
attention = model_output.attentions

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [15]:
#inputs = tokenizer.encode_plus(SMILES_a, return_tensors='pt')
input_ids = encoded_input['input_ids']
#attention = model(input_ids)[-1]
input_id_list = input_ids[0]
tokens = tokenizer.convert_ids_to_tokens(input_id_list)

head_view(attention, tokens)

<IPython.core.display.Javascript object>

Use our generated scripts to convert BPE tokenizer into atomic weights

In [16]:
import token_importance as ti
import process_special as ps
token_scores = ti.token_importance_from_attention(attention)
tokens_clean, scores_clean = ps.remove_special_tokens(
    tokens,
    token_scores.tolist()
)

In [17]:
import viz_token_score as vts

SMILES_a = "C#CC1=CC=CC(N/C2=N/C=N\\C3=CC(OCCOC)=C(OCCOC)C=C23)=C1"

mol, atom_scores, mappings = vts.tokens_scores_to_atom_scores(
    SMILES_a,
    tokens_clean,
    scores_clean,
    agg="max"    # aggregation method when an atom was covered by multiple tokens
)

svg = vts.draw_molecule_attention(SMILES_a, atom_scores, cmap="coolwarm")
vts.save_svg(svg, "viz_atom_attn_a.svg")